# Customer Segmentation & Explainable Churn-Risk Score

**Objective:** group customers using account/behavior features and create an auditable rule-based score.

**Input:** `customer_churn_sample.csv` in the same folder. The observed `Churn` label is excluded from score calculation and segment assignment. It is used only after scoring for descriptive validation. This is a prioritisation exercise, not a predictive ML model.

## 1. Load data and choose features

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("customer_churn_sample.csv")
print("Shape:", df.shape)
display(df.head())
features = ["ContractType", "TenureMonths", "SupportTickets", "MonthlyCharges"]
print("Features used:", features)


## 2. Validate inputs. Missing or invalid feature values are rejected, not silently changed to zero.

In [ ]:
required = ["CustomerID", "ContractType", "TenureMonths", "SupportTickets", "MonthlyCharges"]
assert all(c in df.columns for c in required), "Required columns missing"
assert df[required].notna().all().all(), "Missing required feature values"
assert df["CustomerID"].is_unique, "CustomerID must be unique"
assert (df["TenureMonths"] >= 0).all(), "Negative tenure"
assert (df["SupportTickets"] >= 0).all(), "Negative support tickets"
assert (df["MonthlyCharges"] >= 0).all(), "Negative monthly charges"
assert set(df["ContractType"].unique()).issubset({"Month-to-Month", "One Year", "Two Year"})
print("Input validation passed.")


## 3. Explainable risk score

- Month-to-Month contract: +3
- Tenure ≤ 12 months: +2
- Support tickets ≥ 4: +2
- Monthly charges ≥ ₹80: +1

Risk tiers: Low = 0–2; Medium = 3–4; High = 5–8. `Churn` is not used to calculate the score.

In [ ]:
def calculate_risk(row):
    score, reasons = 0, []
    rules = [
        (row["ContractType"] == "Month-to-Month", 3, "Month-to-Month contract"),
        (row["TenureMonths"] <= 12, 2, "Tenure <= 12 months"),
        (row["SupportTickets"] >= 4, 2, "Support tickets >= 4"),
        (row["MonthlyCharges"] >= 80, 1, "Monthly charges >= ₹80"),
    ]
    for matched, points, reason in rules:
        if matched:
            score += points
            reasons.append(f"{reason} (+{points})")
    return pd.Series({"RiskScore": score, "RiskReasons": "; ".join(reasons) if reasons else "No risk flags"})
df[["RiskScore", "RiskReasons"]] = df.apply(calculate_risk, axis=1)
df["RiskTier"] = df["RiskScore"].apply(lambda x: "Low" if x <= 2 else ("Medium" if x <= 4 else "High"))
display(df[["CustomerID", "RiskScore", "RiskTier", "RiskReasons"]])


## 4. Assign mutually exclusive segments (rules run in this order)

1. **New & High Support:** tenure ≤12 AND support tickets ≥4.
2. **Month-to-Month Watchlist:** Month-to-Month contract, unless already in segment 1.
3. **Established & Stable:** tenure >24 AND support tickets ≤2 AND not Month-to-Month.
4. **Developing Customers:** all remaining customers.

In [ ]:
def assign_segment(row):
    if row["TenureMonths"] <= 12 and row["SupportTickets"] >= 4:
        return "New & High Support"
    if row["ContractType"] == "Month-to-Month":
        return "Month-to-Month Watchlist"
    if row["TenureMonths"] > 24 and row["SupportTickets"] <= 2 and row["ContractType"] != "Month-to-Month":
        return "Established & Stable"
    return "Developing Customers"
df["CustomerSegment"] = df.apply(assign_segment, axis=1)
display(df[["CustomerID", "CustomerSegment", "TenureMonths", "SupportTickets", "ContractType"]])


## 5. Create segment-specific recommendations

In [ ]:
actions = {
    "New & High Support": "Proactive service recovery: contact customer, resolve issues, and offer onboarding help; do not assume a discount is the right fix.",
    "Month-to-Month Watchlist": "Understand why the customer remains monthly; explain annual-plan value and consider a voluntary incentive where appropriate.",
    "Established & Stable": "Maintain service quality, recognise loyalty, and monitor periodically; avoid unnecessary discounting.",
    "Developing Customers": "Use a short satisfaction check-in and monitor service signals; tailor follow-up to customer needs."
}
df["RecommendedAction"] = df["CustomerSegment"].map(actions)
display(df[["CustomerID", "CustomerSegment", "RiskScore", "RiskTier", "RecommendedAction"]])


## 6. Summarise segments and compare observed churn after scoring

In [ ]:
segment_summary = df.groupby("CustomerSegment").agg(
    Customers=("CustomerID", "count"),
    AverageRiskScore=("RiskScore", "mean"),
    HighRiskCustomers=("RiskTier", lambda s: (s == "High").sum()),
    MonthlyCharges=("MonthlyCharges", "sum"),
    ObservedChurnCount=("Churn", lambda s: (s == "Yes").sum())
).reset_index()
segment_summary["ObservedChurnRatePct"] = (100 * segment_summary["ObservedChurnCount"] / segment_summary["Customers"]).round(1)
display(segment_summary.round(2))
display(pd.crosstab(df["RiskTier"], df["Churn"], margins=True))


## 7. Edge-case validation

In [ ]:
def score_values(contract, tenure, tickets, charge):
    return (3 if contract == "Month-to-Month" else 0) + (2 if tenure <= 12 else 0) + (2 if tickets >= 4 else 0) + (1 if charge >= 80 else 0)
assert score_values("Month-to-Month", 12, 4, 80) == 8
assert score_values("Month-to-Month", 13, 3, 79.99) == 3
assert score_values("One Year", 12, 4, 79.99) == 4
assert score_values("One Year", 13, 3, 80) == 1
def tier(s):
    return "Low" if s <= 2 else ("Medium" if s <= 4 else "High")
assert [tier(x) for x in [2,3,4,5]] == ["Low","Medium","Medium","High"]
print("All edge-case tests passed.")


## 8. Export files

In [ ]:
df.to_csv("customer_segments_and_risk_scores.csv", index=False)
segment_summary.to_csv("segment_summary.csv", index=False)
print("Exported customer_segments_and_risk_scores.csv and segment_summary.csv")


## 9. Limitations

This sample contains only 15 synthetic records. The score is a transparent rule-based prioritisation aid, not a probability or validated prediction model. A larger, representative, time-stamped dataset is needed to calibrate thresholds and evaluate predictive performance. The dataset lacks customer feedback, usage trends, complaint severity, competitor offers and intervention outcomes. Recommendations are hypotheses to test, not guaranteed retention outcomes.